In [4]:
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

maze = np.array([
    [0, 0, 0, 0, 0, 0],
    [1, 1, 0, 1, 1, 0],
    [0, 0, 0, 0, 1, 0],
    [0, 1, 1, 0, 1, 0],
    [0, 1, 0, 0, 0, 0],
    [0, 0, 0, 1, 1, 0]
])

start = (0, 0)
goal = (5, 5)
n_rows, n_cols = maze.shape

actions = ['up', 'down', 'left', 'right']

def move(state, action):
    r, c = state
    if action == 'up': r -= 1
    elif action == 'down': r += 1
    elif action == 'left': c -= 1
    elif action == 'right': c += 1

    if r < 0 or r >= n_rows or c < 0 or c >= n_cols or maze[r, c] == 1:
        return state
    return (r, c)

def get_reward(state):
    if state == goal:
        return 100
    return -1

In [5]:
Q = {}
for r in range(n_rows):
    for c in range(n_cols):
        for a in actions:
            Q[((r, c), a)] = 0.0

alpha = 0.1
gamma = 0.9
episodes = 2000

# Episodes at which to "snapshot" the agent's behavior for the animation
snapshot_episodes = [0, 20, 100, 300, 800, 1999]
recorded_paths = {}

for episode in range(episodes):
    # Decay epsilon over time: explore a lot early, less later
    epsilon = max(0.05, 1.0 - episode / (episodes * 0.6))

    state = start
    path = [state]
    steps = 0
    while state != goal and steps < 150:
        if random.random() < epsilon:
            action = random.choice(actions)
        else:
            q_vals = [Q[(state, a)] for a in actions]
            action = actions[np.argmax(q_vals)]

        next_state = move(state, action)
        reward = get_reward(next_state)
        best_next = max(Q[(next_state, a)] for a in actions)
        Q[(state, action)] += alpha * (reward + gamma * best_next - Q[(state, action)])

        state = next_state
        path.append(state)
        steps += 1

    if episode in snapshot_episodes:
        recorded_paths[episode] = path

print("Training done!")
for ep, p in recorded_paths.items():
    reached = "reached goal" if p[-1] == goal else "did not reach goal"
    print(f"Episode {ep}: {len(p)} steps, {reached}")

Training done!
Episode 0: 151 steps, did not reach goal
Episode 20: 62 steps, reached goal
Episode 100: 151 steps, did not reach goal
Episode 300: 37 steps, reached goal
Episode 800: 12 steps, reached goal
Episode 1999: 13 steps, reached goal


In [6]:
# Flatten all snapshot paths into one long sequence, with markers for episode boundaries
all_frames = []  # each entry: (episode_number, step_index_in_path, path)
for ep in snapshot_episodes:
    p = recorded_paths[ep]
    for i in range(len(p)):
        all_frames.append((ep, i, p))

fig, ax = plt.subplots(figsize=(6, 6))

def draw_maze(ax):
    ax.clear()
    ax.imshow(maze, cmap='binary')
    ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
    ax.grid(which='minor', color='gray', linewidth=1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.scatter(start[1], start[0], color='green', s=200, marker='s')
    ax.scatter(goal[1], goal[0], color='red', s=200, marker='*')

def update(frame_idx):
    ep, i, p = all_frames[frame_idx]
    draw_maze(ax)
    trail = np.array(p[:i+1])
    ax.plot(trail[:, 1], trail[:, 0], color='blue', alpha=0.4, linewidth=2)
    r, c = p[i]
    ax.scatter(c, r, color='blue', s=150, marker='o')
    steps_taken = len(p) - 1
    ax.set_title(f"Episode {ep}  |  step {i+1}/{len(p)}  |  total steps this run: {steps_taken}")

ani = FuncAnimation(fig, update, frames=len(all_frames), interval=150, repeat=False)
plt.close()
HTML(ani.to_jshtml())